In [ ]:
# =====================================================================
# 1: IMPORTS
# =====================================================================
import os
import multiprocessing
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from Library.config import BASE_PARAMS
from Library import run_trajectory_grid

# =====================================================================
# 2: CORE FUNCTION 
# =====================================================================
def RunGammaSearch(df_patient, neut_col, blast_col, min_time=30):
    # Filter missing clinical values to prevent NaN propagation
    df_clean = df_patient.dropna(subset=[neut_col, blast_col]).copy()
    
    # Standardize neutrophil counts (per kg)
    target_neutro = (df_clean[neut_col].to_numpy(dtype=float) * 5 * 10**9 / 80)[:, None]
    target_blasts = df_clean[blast_col].to_numpy(dtype=float)[:, None]
    
    if len(df_clean) == 0:
        raise ValueError("No valid patient records found after dropping NaNs.")
    
    phsc_range = np.arange(0.95, 1.001, 0.01)
    coef_range = np.arange(0.25, 30.05, 0.1)

    
    #gamma_low = np.logspace(-8.0, -6.5, num=2, endpoint=False)
    gamma_focus = np.logspace(-8.0, -5.5, num=5, endpoint=True)
    gamma_high = np.logspace(-5.3, -3.0, num=2, endpoint=True)
    gamma_test_range = np.unique(np.concatenate(( gamma_focus, gamma_high)))
    
    print(f"Total unique gamma parameters to test: {len(gamma_test_range)}")
    
    gamma_optimization_results = []
    y0_healthy = [2.42105 * 10**11, 4 * 10**8]
    neut_range = y0_healthy[1]
    
    n_cores = multiprocessing.cpu_count()
    n_chunks = min(n_cores, len(phsc_range))
    phsc_chunks = np.array_split(phsc_range, n_chunks)

    print(f"--- STARTING PARALLEL GRID SWEEP ACROSS {len(gamma_test_range)} BOUNDARIES (min_time={min_time}) ---")
    for g_val in gamma_test_range:
        print(f" Evaluating Parameter Branch: gamma = {g_val:.2e} ...")
        
        p = BASE_PARAMS.copy()
        p['gamma'] = g_val
        
        # Parallel ODE trajectory generation
        chunked_results = Parallel(n_jobs=n_chunks)(
            delayed(run_trajectory_grid.run_trajectory_grid)(
                chunk, coef_range, p, g_val, y0_healthy, min_time=min_time
            ) for chunk in phsc_chunks
        )
        
        # Flatten trajectory list
        valid_sim_trajectories = [df for sublist in chunked_results for df in sublist]

        if not valid_sim_trajectories:
            print(f"  -> Notice: Zero trajectories satisfied timeline filters at gamma={g_val:.2e}. Skipping.")
            continue
            
        all_sim_blasts = np.concatenate([df['BlastPerc'].values * 100 for df in valid_sim_trajectories])
        all_sim_neuts  = np.concatenate([df['Neut'].values for df in valid_sim_trajectories])
        
        blast_err = (all_sim_blasts - target_blasts) / 100.0
        neutro_err = (all_sim_neuts - target_neutro) / neut_range
        
        # Find minimum distance per patient across all simulation points
        combined_err_matrix = np.sqrt(blast_err**2 + neutro_err**2)
        patient_min_errors = np.min(combined_err_matrix, axis=1)
        
        global_mean_residual = float(np.mean(patient_min_errors))
        print(f"  -> Global Error: {global_mean_residual:.4f}")
        
        gamma_optimization_results.append({
            'gamma_val': g_val,
            'log_gamma': np.log10(g_val),
            'global_error': global_mean_residual
        })

    return pd.DataFrame(gamma_optimization_results)

# =====================================================================
# 3: DATA LOADING 
# =====================================================================
venex_df = pd.read_excel("InputData/venex_cohort_OS_labs_with_mutations.xlsx")
moffitt_df = pd.read_excel("InputData/moffitt_cohort_OS_labs.xlsx")

# =====================================================================
# 4: EXECUTE SEARCHES
# =====================================================================
print("Running Unified VenEx Cohort...")
venex_results = RunGammaSearch(venex_df, 'b_neut', 'bm_blasts', min_time=30)

print("\nRunning Unified Moffitt Cohort...")
moffitt_results = RunGammaSearch(moffitt_df, 'ANC at Dx', 'BM Blasts at Dx by flow', min_time=30)

# =====================================================================
# 5: SAVE RESULTS
# =====================================================================
os.makedirs("IntermediateResults", exist_ok=True)

venex_results.to_csv("IntermediateResults/venex_gamma_errors.csv", index=False)
moffitt_results.to_csv("IntermediateResults/moffitt_gamma_errors.csv", index=False)

print("\nAll gamma searches completed and saved successfully!")